# Inspect silver tables

In [0]:
%sql
SELECT * FROM workspace.silver.crm_products LIMIT 10

In [0]:
%sql
SELECT * FROM workspace.silver.erp_category LIMIT 10

# Join table

In [0]:
import pyspark.sql.functions as F

In [0]:
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA silver")

query = """
SELECT
  p.product_id AS product_id,
  COALESCE(p.product_key, "Unknow") AS product_key,
  COALESCE(p.product_name, "Unknow") AS product_name,
  p.product_cost,
  COALESCE(p.product_line, "Unknow") AS product_line,
  COALESCE(c.category_key, "Unknow") AS category_key,
  COALESCE(c.category_name, "Unknow") AS category_name,
  COALESCE(c.subcategory_name, "Unknow") AS subcategory_name,
  c.maintenance,
  p.start_date,
  p.end_date
FROM crm_products p
LEFT JOIN erp_category c ON p.category_key = c.category_key
"""

df = spark.sql(query)

df.display()
df.printSchema()

In [0]:
df.select([
  F.sum(F.when(F.col(c).isNull(), F.lit(1)).otherwise(F.lit(0))).alias(c)
  for c in df.columns
]).display()

# Write table

In [0]:
df.write.mode("overwrite").saveAsTable("workspace.gold.products")

## Sanity check

In [0]:
%sql
SELECT * FROM workspace.gold.products LIMIT 5